In [ ]:
### Assignment 1



1. 
2. 
3. 
4. Create a graph, depicting the result of (3). Set the parameters: what the color and the size of the nodes and the width of the edges stands for. Choose a directed or a non-directed graph. Describe and justify your choices of the parameters. -- 2 points
5. Perform network analysis of the graph. Mesure the density, split the graph into communities etc. Describe and justify what and why you are doing. -- 2 points
6. Could you draw any meaningful conclusions from your analysis? Provide a short (a couple of sentences) answer. -- 1 point

# Homework 1

In [46]:
!pip install tqdm spacy --upgrade jupyter ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 1.1 MB/s eta 0:00:00a 0:00:01
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 1.3 MB/s eta 0:00:0000:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.8/260.8 kB 1.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/134.1 kB 1.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 3.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 4.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 4.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9

In [48]:
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 3.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 673.0 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 2.1 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')


In [49]:
from tqdm.notebook import tqdm
import re
import spacy

nlp = spacy.load("ru_core_news_sm")

## Choose a long text, a novel, in Russian or in English

I chose the novel "Childhood" by Leo Tolstoy. I chose a Leo Tolstoy novel specifically because he is notorious for writing long, complex sentences, on which it would be curious to perform network analysis.

In [1]:
!wget https://royallib.com/get/txt/tolstoy_lev/detstvo.zip

--2026-09-19 21:12:52--  https://royallib.com/get/txt/tolstoy_lev/detstvo.zip
Resolving royallib.com (royallib.com)... 45.9.73.94
Connecting to royallib.com (royallib.com)|45.9.73.94|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 83035 (81K) [application/octet-stream]
Saving to: ‘detstvo.zip’

detstvo.zip         100%[===================>]  81,09K  --.-KB/s    in 0,1s    

2026-09-19 21:12:53 (738 KB/s) - ‘detstvo.zip’ saved [83035/83035]



In [ ]:
!unzip detstvo.zip -d data
!rm detstvo.zip

In [ ]:
with open('data/Толстой Лев. Детство - royallib.ru.txt',
          encoding='Windows-1252') as f:
    book = f.read().encode('cp1252').decode('cp1251')
book = book.split('\n\n')

### Clearing the novel

* deleting the technical text like chapter names, empty lines
* deleting special symbols
* deleting Russian translations of the phrases in foreign languages, since the translation was written by an editor and not by the author

In [83]:
lines_to_avoid = [0, 1, 2, 3, 4, 5, 6]
for line in range(len(book)):
    if book[line] == '':
        lines_to_avoid.append(line)
    if book[line][:5] == 'Глава':
        lines_to_avoid.append(line)
        lines_to_avoid.append(line + 1)
    if line >= 722:
        lines_to_avoid.append(line)
clear_book = []
for line in range(len(book)):
    if line not in lines_to_avoid:
        book_line = book[line].replace('\xa0', ' ').replace('\t\t', '').replace('\n', ' ')
        book_line = re.sub('\\[.*\\]', '',  book_line)
        clear_book.append(book_line)

print('Before:', '\n', '\n'.join(book[5:15]), '\n\n',
      'After:', '\n', '\n'.join(clear_book[:5]))

clear_book = ' '.join(clear_book[:100])

Before: 
 
Лев Николаевич Толстой
Детство

Глава I.
УЧИТЕЛЬ КАРЛ ИВАНЫЧ
12 августа 18..., ровно в третий день после дня моего рождения, в который мне минуло десять лет и в который я получил такие чудесные подарки, в семь часов утра — Карл Иваныч разбудил меня, ударив над самой моей головой хлопушкой — из сахарной бумаги на палке — по мухе. Он сделал это так неловко, что задел образок моего ангела, висевший на дубовой спинке кровати, и что убитая муха упала мне прямо на голову. Я высунул нос из-под одеяла, остановил рукою образок, который продолжал качаться, скинул убитую муху на пол и хотя заспанными, но сердитыми глазами окинул Карла Иваныча. Он же, в пестром ваточном халате, подпоясанном поясом из той же материи, в красной вязаной ермолке с кисточкой и в мягких козловых сапогах, продолжал ходить около стен, прицеливаться и хлопать.
«Положим, — думал я, — я маленький, но зачем он тревожит меня? Отчего он не бьет мух около Володи ной постели? вон их сколько! Нет, Володя старше меня; а 

## Lemmatize the novel. -- 1 point

I am also removing punctuation, stopwords, and digits from the text. The output is a list of lemmas from the entire novel

In [92]:
d = nlp(clear_book)
lemmatized_novel = []
for token in d:
    if token.pos_ != 'PUNCT' and not token.is_stop and not token.is_digit:
        lemmatized_novel.append(token.lemma_)
    #print(token.text, token.lemma_, token.pos_, token.is_stop)
lemmatized_novel

['август',
 'ровно',
 'третий',
 'день',
 'день',
 'рождение',
 'минуть',
 'десять',
 'год',
 'получить',
 'чудесный',
 'подарок',
 'семь',
 'час',
 'утро',
 'карл',
 'иванович',
 'разбудить',
 'ударить',
 'голова',
 'хлопушка',
 'сахарный',
 'бумага',
 'палка',
 'муха',
 'сделать',
 'неловкий',
 'задеть',
 'образок',
 'ангел',
 'висеть',
 'дубовый',
 'спинка',
 'кровать',
 'убить',
 'муха',
 'упасть',
 'прямо',
 'голова',
 'высунуть',
 'нос',
 '-',
 'одеяло',
 'остановить',
 'рукою',
 'образок',
 'продолжать',
 'качаться',
 'скинуть',
 'убить',
 'муха',
 'пол',
 'заспанный',
 'сердитый',
 'глаз',
 'окинуть',
 'карл',
 'иванович',
 'пёстрый',
 'ваточный',
 'халат',
 'подпоясать',
 'пояс',
 'материя',
 'красный',
 'вязаный',
 'ермолка',
 'кисточка',
 'мягкий',
 'козловый',
 'сапог',
 'продолжать',
 'ходить',
 'стена',
 'прицеливаться',
 'хлопать',
 'положить',
 'думать',
 'маленький',
 'тревожить',
 'бить',
 'муха',
 'володи',
 'ной',
 'постель',
 'володя',
 'старший',
 'мучить',
 'дума

## Choose a word and extract all its children (all its syntactic dependencies) and all the children of its children (secondary dependencies). -- 2 points

I will firstly find the longest and most interesting sentences in the novel and work with one of them.

In [116]:
lengths = []
sentences = []
for sent in d.sents:
    sentences.append(sent.text)
    s_len = 0
    for token in sent:
        s_len += 1
    lengths.append(s_len)

sentences_and_lengths = dict(zip(sentences, lengths))
sentences_and_lengths = {k: v for k, v in sorted(sentences_and_lengths.items(),
                                                  key=lambda item: item[1],
                                                  reverse=True)}

In [123]:
list(sentences_and_lengths.keys())[0]

'Бывало, стоишь, стоишь в углу, так что колени и спина заболят, и думаешь: «Забыл про меня Карл Иваныч: ему, должно быть, покойно сидеть на мягком кресле и читать свою гидростатику, — а каково мне?» — и начнешь, чтобы напомнить о себе, потихоньку отворять и затворять заслонку или ковырять штукатурку со стены; но если вдруг упадет с шумом слишком большой кусок на землю — право, один страх хуже всякого наказания.'

In [124]:
d = nlp(list(sentences_and_lengths.keys())[0])
for token in d:
    print(token.text, token.dep_, token.head.text, token.head.pos_,
            [child for child in token.children])



Бывало ROOT Бывало VERB [стоишь, стоишь, заболят, .]
, punct стоишь VERB []
стоишь conj Бывало VERB [,]
, punct стоишь VERB []
стоишь conj Бывало VERB [,, углу]
в case углу NOUN []
углу obl стоишь VERB [в]
, punct заболят VERB []
так mark заболят VERB [что]
что fixed так ADV []
колени nsubj заболят VERB [спина]
и cc спина NOUN []
спина conj колени NOUN [и]
заболят advcl Бывало VERB [,, так, колени, думаешь]
, punct думаешь VERB []
и cc думаешь VERB []
думаешь conj заболят VERB [,, и, Забыл]
: punct Забыл VERB []
« punct Забыл VERB []
Забыл parataxis думаешь VERB [:, «, меня, Карл, сидеть, каково, ?, », начнешь, упадет]
про case меня PRON []
меня obl Забыл VERB [про]
Карл nsubj Забыл VERB [Иваныч]
Иваныч flat:name Карл PROPN []
: punct сидеть VERB []
ему iobj покойно ADV []
, punct должно ADJ []
должно parataxis покойно ADV [,, быть, ,]
быть xcomp должно ADJ []
, punct должно ADJ []
покойно advmod сидеть VERB [ему, должно]
сидеть parataxis Забыл VERB [:, покойно, кресле, читать]
на case